# Reproducibility Notebook
## LLM-Based Semantic Capability Mapping for Heterogeneous Smart-Home IoT Devices

This notebook reproduces the Mistral Small 4 experiment. The benchmark and Gold Standard were frozen before model execution.


In [ ]:
!git clone -q https://github.com/Ing-Armando-Ortiz/smart-home-capability-mapping.git
%cd smart-home-capability-mapping
!pip install -q -r requirements.txt


### API key
Create a Colab Secret named `MISTRAL_API_KEY`. The key is never written to the notebook or repository.

In [ ]:
import os
from google.colab import userdata
os.environ['MISTRAL_API_KEY'] = userdata.get('MISTRAL_API_KEY')
assert os.environ['MISTRAL_API_KEY'], 'Missing Colab Secret: MISTRAL_API_KEY'
print('API key loaded from Colab Secrets.')


## Environment record
Record package and Python versions before inference.

In [ ]:
import platform
print(platform.python_version())
!pip freeze | sort | tee environment_freeze.txt


## Integrity checks
Verify the identity-free execution payload and experiment size.

In [ ]:
import csv
rows = list(csv.DictReader(open('data/execution_payload_v1.3.csv', encoding='utf-8')))
assert len(rows) == 324
assert set(r['strategy_id'] for r in rows) == {'S1','S2','S3'}
assert all('gold' not in k.lower() for k in rows[0].keys())
print('Integrity checks passed:', len(rows), 'planned calls.')


## Dry run — 9 calls
**Not used for paper scoring.** Confirms authentication, output parsing, tokens and structured-output plumbing.

In [ ]:
!python scripts/run_experiment.py --dry-run


In [ ]:
import pandas as pd
dry = pd.read_csv('results/raw/dry_run_mistral_small_2603.csv')
display(dry[['run_id','representation','strategy_id','predicted_capabilities','json_valid','schema_valid','latency_ms','status']])
assert (dry['status'] == 'DONE').all(), 'Dry run contains API errors.'


## Formal experiment — 324 scored calls
Run this cell only after the dry run succeeds.

In [ ]:
!python scripts/run_experiment.py


## Evaluation


In [ ]:
!python scripts/evaluate.py
summary = pd.read_csv('results/processed/summary_metrics.csv')
display(summary)


## Save artifacts
Download the raw and processed outputs or commit an archival copy after checking that no credentials are present.

In [ ]:
from google.colab import files
files.download('results/raw/mistral_small_2603.csv')
files.download('results/processed/summary_metrics.csv')
files.download('environment_freeze.txt')
